# VanaDrishti on Colab — 00: download BraDD-S1TS and inspect (Phase 0)

Thin driver: every cell calls code that lives in the repo (`src/`, `scripts/`), so the notebook stays in sync with the tested code. Outputs go to `MyDrive/vanadrishti/`.

## 1. Setup (run every session)

Before running: **Runtime → Change runtime type → T4 GPU** (any GPU works).
Colab forgets everything except Google Drive when the session ends, so the code and the dataset are
re-fetched each session and all run outputs (checkpoints, logs, metrics) are written to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/vana-drishti'
BRANCH = 'claude/new-session-uto7x0'
DRIVE = '/content/drive/MyDrive/vanadrishti'
RUNS = f'{DRIVE}/runs'
STATS = f'{DRIVE}/stats/vanadrishti_close_train_stats.pt'
!mkdir -p {RUNS} {DRIVE}/stats {DRIVE}/results

In [ ]:
import os
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/krish-k1301/vana-drishti.git {REPO}
%cd {REPO}
!git log --oneline -1

Pinned versions from `environment.yml`. torch 2.5.1 is required: the upstream reference code calls
`torch.load` in a way that breaks on torch >= 2.6. pip may print dependency-conflict warnings about
other Colab packages; they don't affect this project. All training runs as `!python ...` subprocesses,
so no runtime restart is needed after the install.

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q lightning==2.4.0 torchmetrics==1.9.0 numpy==2.1.3 pandas==2.2.3 scipy==1.17.1 pyyaml==6.0.3 einops==0.8.2 matplotlib==3.11.2 tqdm==4.70.1 requests==2.34.2 pytest==9.1.1
!python -c "import torch, lightning; print(torch.__version__, lightning.__version__, 'CUDA:', torch.cuda.is_available())"

## 2. Dataset

BraDD-S1TS from Zenodo: a 17.7 GB zip, md5-checked. The **first** session downloads it and keeps a copy on
Drive (`MyDrive/vanadrishti/data/`); every later session copies that one file back (a few minutes) instead of
downloading. The data is always **unzipped to Colab's local disk**: training reads ~26,000 small files, which
is far too slow from Drive. Skipped entirely if the data is already unzipped in this session.

In [ ]:
import glob
ZIP_NAME = 'BraDD-S1TS_zenodo.zip'
ZIP_DRIVE = f'{DRIVE}/data/{ZIP_NAME}'
ZIP_LOCAL = f'{REPO}/data/{ZIP_NAME}'

def find_root():
    hits = sorted(glob.glob(f'{REPO}/data/**/meta.csv', recursive=True))
    return os.path.dirname(hits[0]) if hits else None

if find_root() is None:
    !mkdir -p {REPO}/data {DRIVE}/data
    !df -h /content | tail -1
    if os.path.exists(ZIP_DRIVE):
        print('copying the cached zip from Drive ...')
        !cp {ZIP_DRIVE} {ZIP_LOCAL}
        !python scripts/download_bradd.py --config configs/data/downloads.yaml --delete-zip
    else:
        !python scripts/download_bradd.py --config configs/data/downloads.yaml
        print('caching the verified zip on Drive for the next sessions ...')
        !cp {ZIP_LOCAL} {ZIP_DRIVE} && rm {ZIP_LOCAL}
ROOT = find_root()
print('BraDD root:', ROOT)
assert ROOT is not None, 'meta.csv not found: check the output above'

## 3. Phase 0 inspection

Answers the PRD Phase 0 questions: every file in the archive, does anything geolocate the patches,
does `close_stats.pt` ship, keys/shapes/dB ranges, T distribution, label-date spacing, split counts
vs the paper. The report is copied to Drive.

In [ ]:
!python scripts/inspect_bradd.py --config configs/data/bradd.yaml --root {ROOT} --out-dir {DRIVE}/results/phase0 --n-samples -1

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{DRIVE}/results/phase0/report.md').read()))

If the report says `close_stats.pt` **ships** in the archive, both Phase 1 tracks must normalise with
the same file: in notebook 01 set `STATS` to that shipped file (path printed in the report).